In [1]:
from google.colab import files
uploaded = files.upload()

Saving IMDB_Dataset.csv to IMDB_Dataset.csv


In [3]:
import pandas as pd
df = pd.read_csv("IMDB_Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
print("Shape of dataset:", df.shape)

# printing column names
print("\nColumns:")
print(df.columns)

# checking class distribution
print("\nClass Distribution:")
print(df['sentiment'].value_counts())

Shape of dataset: (50000, 2)

Columns:
Index(['review', 'sentiment'], dtype='object')

Class Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [8]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

nltk.download('stopwords')
nltk.download('punkt')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)

    # tokenization
    words = text.split()

    # remove stopwords
    words = [w for w in words if w not in stop_words]

    # stemming
    words = [stemmer.stem(w) for w in words]

    return " ".join(words)

# This line was moved outside the function definition
df['processed_text'] = df['review'].apply(preprocess_text)
df[['review', 'processed_text']].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


,review,processed_text
0,One of the other reviewers has mentioned that ...,one review mention watch oz episod youll hook ...
1,A wonderful little production. <br /><br />The...,wonder littl product br br film techniqu unass...
2,I thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,basic there famili littl boy jake think there ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visual stun film...


In [10]:
# input features (text)
X = df['processed_text']

# target labels
y = df['sentiment']
# converting text into numerical features using Bag of Words
from sklearn.feature_extraction.text import CountVectorizer

# creating BoW model
bow = CountVectorizer()

# converting text into vectors
X_bow = bow.fit_transform(X)

print("BoW Shape:", X_bow.shape)

BoW Shape: (50000, 137463)


In [11]:
#converting text using TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# creating TF-IDF model
tfidf = TfidfVectorizer()
# transforming text
X_tfidf = tfidf.fit_transform(X)
print("TF-IDF Shape:", X_tfidf.shape)

TF-IDF Shape: (50000, 137463)


In [12]:
#Logistic Regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Calculate and print the accuracy score
accuracy = accuracy_score(y_test, y_pred_lr)
print(f"Accuracy of Logistic Regression model: {accuracy:.4f}")

Accuracy of Logistic Regression model: 0.8898


In [13]:
# Naive Bayes
from sklearn.naive_bayes import MultinomialNB
nb = MultinomialNB()
nb.fit(X_train, y_train)

MultinomialNB()

In [16]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

DecisionTreeClassifier()

In [17]:
from sklearn.metrics import accuracy_score, classification_report
y_pred_lr = lr.predict(X_test)
print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

# Naive Bayes evaluation
y_pred_nb = nb.predict(X_test)

print("\nNaive Bayes")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

# Decision Tree evaluation
y_pred_dt = dt.predict(X_test)

print("\nDecision Tree")
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

Logistic Regression
Accuracy: 0.8898
              precision    recall  f1-score   support

    negative       0.90      0.87      0.89      4961
    positive       0.88      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000


Naive Bayes
Accuracy: 0.8622
              precision    recall  f1-score   support

    negative       0.85      0.88      0.86      4961
    positive       0.88      0.85      0.86      5039

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000


Decision Tree
Accuracy: 0.7181
              precision    recall  f1-score   support

    negative       0.71      0.72      0.72      4961
    positive       0.72      0.72      0.72      5039

    accuracy                           0.72     10000
   macro avg       0.72      

6. Comparision and Insights
Logistic Regression performed best. TF-IDF worked better than Bag of Words. Preprocessing improved model accuracy. Naive Bayes was fast. Decision Tree slightly overfitted.

Overall, Logistic Regression with TF-IDF is the best choice for this task.

Conclusion:
This project demonstrates how text data can be processed and used to build sentiment analysis models. Different preprocessing techniques and feature engineering methods were applied. Multiple models were trained and compared. Among all models, Logistic Regression performed best with TF-IDF features.